# EcoGraphRAG — Notebook 5: Ablation Study

**Goal:** K-sensitivity test — how many graph chunks (k) to include at BFS depth=1.

**Baseline results:**
| Method | EM | F1 |
|--------|-----|-----|
| BM25 | 20.6% | 37.7% |
| Graph-RAG (default, depth=2, k=5) | 21.2% | 38.8% |
| Flat RAG (FAISS-only) | **26.0%** | **45.3%** |
| Ablation B (depth=1, k=5) | 25.0% | 44.7% |
| Ablation E (depth=1, k=3, FAISS-heavy) | 25.4% | **45.9%** |

**New configs:** k=2 and k=3 at depth=1, 0.5/0.5 weights (same as B, varying only k)

**Outputs:** `ablation_k_sensitivity.json`

## 1. Setup

In [ ]:
from google.colab import drive
import os, json, time, re, pickle, gc
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT+'data/'; DRIVE_INDICES = DRIVE_ROOT+'indices/'
DRIVE_OUTPUTS = DRIVE_ROOT+'outputs/'; DRIVE_CHECKPOINTS = DRIVE_ROOT+'checkpoints/'
for d in [DRIVE_OUTPUTS,DRIVE_CHECKPOINTS]: os.makedirs(d,exist_ok=True)
print('Drive mounted.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu spacy networkx
!python -m spacy download en_core_web_sm -q

## 2. Load Data + Precompute Embeddings

In [ ]:
import torch, numpy as np, faiss, spacy
from collections import defaultdict, deque
from sentence_transformers import SentenceTransformer

with open(DRIVE_DATA+'hotpotqa_500.json') as f: questions=json.load(f)['questions']
with open(DRIVE_DATA+'chunks.json') as f: chunks=json.load(f)
chunk_lookup = {c['chunk_id']:c for c in chunks}
chunk_mapping = [c['chunk_id'] for c in chunks]
index = faiss.read_index(DRIVE_INDICES+'faiss_index.bin')
with open(DRIVE_INDICES+'entity_graph.gpickle','rb') as f: graph=pickle.load(f)
nlp = spacy.load('en_core_web_sm',disable=['tagger','parser','lemmatizer'])

# Precompute query embeddings, then delete model
EMBEDDING_MODEL = 'nomic-ai/nomic-embed-text-v1'
embed_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True, device='cpu')
print('Encoding 500 query embeddings...')
all_query_embs = embed_model.encode(
    ['search_query: '+q['question'] for q in questions],
    normalize_embeddings=True, batch_size=64, show_progress_bar=True
).astype(np.float32)
del embed_model; gc.collect()
print(f'Done. Qs:{len(questions)} Chunks:{len(chunks)} FAISS:{index.ntotal} Graph:{graph.number_of_nodes()}n/{graph.number_of_edges()}e')

## 3. Retrieval Functions (Configurable)

In [ ]:
def faiss_retrieve(question_idx, top_k=5):
    q_emb = all_query_embs[question_idx:question_idx+1]
    sc, ix = index.search(q_emb, top_k)
    return [dict(chunk_lookup[chunk_mapping[i]], score=float(s))
            for s,i in zip(sc[0],ix[0]) if chunk_mapping[i] in chunk_lookup]

def extract_query_entities(question):
    doc = nlp(question)
    seen, ents = set(), []
    for e in doc.ents:
        k = e.text.strip().lower()
        if len(k)>=2 and k not in seen: seen.add(k); ents.append(k)
    return ents

def graph_bfs(q_ents, depth=2, top_k=5):
    visited, cs, nt = set(), defaultdict(float), 0
    for ent in q_ents:
        if ent not in graph: continue
        q, lv = deque([(ent,0)]), set()
        while q:
            nd,d = q.popleft()
            if nd in lv: continue
            lv.add(nd)
            if nd not in visited:
                visited.add(nd); nt += 1
                for cid in graph.nodes[nd].get('chunk_ids',[]):
                    cs[cid] += 1.0/(1.0+d)
            if d < depth:
                for nb in graph.neighbors(nd):
                    if nb not in lv: q.append((nb,d+1))
    ranked = sorted(cs.items(), key=lambda x:x[1], reverse=True)
    return [c for c,_ in ranked[:top_k]], nt

def merged_retrieve(question, question_idx, cfg):
    '''Configurable merged retrieval.'''
    qe = extract_query_entities(question)
    
    # FAISS
    if cfg['faiss_weight'] > 0:
        fr = faiss_retrieve(question_idx, top_k=cfg.get('faiss_top_k', 5))
    else:
        fr = []
    
    # Graph BFS
    if cfg['graph_weight'] > 0:
        gi, nt = graph_bfs(qe, depth=cfg['bfs_depth'], top_k=cfg['graph_top_k'])
    else:
        gi, nt = [], 0
    
    # Merge
    scores, sources = defaultdict(float), defaultdict(list)
    if fr:
        mx,mn = max(r['score'] for r in fr), min(r['score'] for r in fr)
        rng = mx-mn if mx!=mn else 1.0
        for r in fr:
            scores[r['chunk_id']] += cfg['faiss_weight']*((r['score']-mn)/rng)
            sources[r['chunk_id']].append('faiss')
    for rank,cid in enumerate(gi):
        scores[cid] += cfg['graph_weight']*(1.0/(1.0+rank))
        sources[cid].append('graph')
    
    ranked = sorted(scores.items(), key=lambda x:x[1], reverse=True)
    merged = []
    for cid,sc in ranked[:cfg['merged_top_k']]:
        if cid in chunk_lookup:
            c = dict(chunk_lookup[cid])
            c['merged_score']=round(sc,4); c['source']='+'.join(sources[cid])
            merged.append(c)
    return merged, qe, nt

print('Retrieval functions ready.')

## 4. Load Mistral-7B

In [ ]:
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.float16)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f'Model loaded. GPU: {torch.cuda.max_memory_allocated()/(1024**2):.0f} MB')

In [ ]:
PROMPT = '<s>[INST] Answer the following question using ONLY the provided context.\nGive a short, precise answer (1-5 words). Do not explain.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer: [/INST]'

def generate_answer(question, ctx_chunks):
    ctx = '\n\n'.join([c['text'] for c in ctx_chunks])
    prompt = PROMPT.format(context=ctx, question=question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k:v.to(model.device) for k,v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                             temperature=0.1, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

## 5. K-Sensitivity Configs (depth=1, weights=0.5/0.5)

| Config | Graph Top-K | Controls |
|--------|-------------|----------|
| K2 | 2 | Minimal graph signal |
| K3 | 3 | Moderate graph signal |
| *(B: already done)* | *5* | *Heavy graph signal* |

Combined with existing B (k=5), this gives a clean k={2,3,5} sweep.

In [ ]:
ABLATION_CONFIGS = {
    'K2_depth1': {
        'bfs_depth': 1, 'graph_top_k': 2, 'graph_weight': 0.5,
        'faiss_weight': 0.5, 'merged_top_k': 6, 'faiss_top_k': 5,
        'desc': 'depth=1, k=2 (minimal graph)'
    },
    'K3_depth1': {
        'bfs_depth': 1, 'graph_top_k': 3, 'graph_weight': 0.5,
        'faiss_weight': 0.5, 'merged_top_k': 6, 'faiss_top_k': 5,
        'desc': 'depth=1, k=3 (moderate graph)'
    },
}

print(f'{len(ABLATION_CONFIGS)} configs to run (k=5 already done as Config B).')
for name, cfg in ABLATION_CONFIGS.items():
    print(f'  {name}: {cfg["desc"]}')

## 6. Run All Ablations (500 Qs × 2 configs)

In [ ]:
import string
from collections import Counter

def normalize(s):
    s=s.lower(); s=re.sub(r'\b(a|an|the)\b',' ',s)
    s=''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())

def em_score(p,g): return int(normalize(p)==normalize(g))
def f1_calc(p,g):
    pt,gt = normalize(p).split(), normalize(g).split()
    if not pt or not gt: return float(pt==gt)
    c = sum((Counter(pt)&Counter(gt)).values())
    if c==0: return 0.0
    return 2*(c/len(pt))*(c/len(gt))/(c/len(pt)+c/len(gt))

ALL_RESULTS_FILE = DRIVE_OUTPUTS + 'ablation_k_sensitivity.json'
CKPT_FILE = DRIVE_CHECKPOINTS + 'ablation_k_checkpoint.json'

# Load checkpoint if exists
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f: ckpt = json.load(f)
    all_ablation_results = ckpt['all_results']
    current_config = ckpt['current_config']
    current_idx = ckpt['current_index'] + 1
    print(f'Resuming from config={current_config}, idx={current_idx}')
else:
    all_ablation_results = {}
    current_config = None
    current_idx = 0

total = len(questions)
started = current_config is None

for config_name, cfg in ABLATION_CONFIGS.items():
    # Skip already completed configs
    if not started:
        if config_name == current_config:
            started = True
        else:
            continue
    
    # Initialize results for this config
    if config_name not in all_ablation_results:
        all_ablation_results[config_name] = []
        start_idx = 0
    else:
        start_idx = current_idx if config_name == current_config else 0
    
    results = all_ablation_results[config_name]
    print(f'\n{"="*60}')
    print(f'Config: {config_name} — {cfg["desc"]}')
    print(f'  BFS={cfg["bfs_depth"]} GraphK={cfg["graph_top_k"]} GW={cfg["graph_weight"]} FW={cfg["faiss_weight"]}')
    print(f'  Running {start_idx} to {total-1}...')
    print(f'{"="*60}')
    
    for i in range(start_idx, total):
        q = questions[i]
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        ctx, qe, nt = merged_retrieve(q['question'], i, cfg)
        pred = generate_answer(q['question'], ctx)
        lat = time.time() - t0
        gpu = torch.cuda.max_memory_allocated()/(1024**2)
        
        results.append({
            'id': q['id'], 'question': q['question'],
            'prediction': pred, 'gold': q['answer'],
            'type': q['type'], 'level': q.get('level','unknown'),
            'latency_seconds': round(lat,3), 'peak_gpu_mb': round(gpu,2),
            'chunks_retrieved': len(ctx), 'nodes_traversed': nt,
            'query_entities': qe, 'config': config_name
        })
        
        if (i+1)%25==0 or i==total-1:
            print(f'  [{i+1}/{total}] {lat:.2f}s nodes={nt} P:"{pred[:30]}" G:"{q["answer"]}')
            # Checkpoint
            all_ablation_results[config_name] = results
            with open(CKPT_FILE,'w') as f:
                json.dump({'current_config':config_name, 'current_index':i, 'all_results':all_ablation_results}, f, ensure_ascii=False)
            gc.collect(); torch.cuda.empty_cache()
    
    # Quick metrics for this config
    em_s = [em_score(r['prediction'],r['gold']) for r in results]
    f1_s = [f1_calc(r['prediction'],r['gold']) for r in results]
    print(f'\n  >> {config_name}: EM={sum(em_s)/len(em_s)*100:.1f}% F1={sum(f1_s)/len(f1_s)*100:.1f}%')
    
    # Reset for next config
    current_idx = 0

# Save final
with open(ALL_RESULTS_FILE,'w') as f:
    json.dump(all_ablation_results, f, indent=2, ensure_ascii=False)
if os.path.exists(CKPT_FILE): os.remove(CKPT_FILE)
print(f'\n\nAll ablation results saved to {ALL_RESULTS_FILE}')

## 7. Summary: All Results Comparison

In [ ]:
# Reference results
baselines = {
    'BM25': {'em': 0.2060, 'f1': 0.3772},
    'Flat_RAG': {'em': 0.2600, 'f1': 0.4534},
    'Graph_RAG_default (d=2,k=5)': {'em': 0.2120, 'f1': 0.3880},
    'B_depth1_k5 (previous)': {'em': 0.2500, 'f1': 0.4470},
    'E_best_combo (previous)': {'em': 0.2540, 'f1': 0.4590},
}

print(f'\n{"="*70}')
print(f'{"K-SENSITIVITY ANALYSIS (depth=1, weights=0.5/0.5)":^70}')
print(f'{"="*70}')
print(f'{"Config":<35} {"EM":>8} {"F1":>8} {"Br_EM":>8} {"Br_F1":>8} {"Co_EM":>8} {"Co_F1":>8}')
print('-'*70)

# Print baselines
for name, m in baselines.items():
    print(f'{name:<35} {m["em"]*100:>7.1f}% {m["f1"]*100:>7.1f}%')
print('-'*70)

# Print new k-sensitivity results
summary_metrics = {}
for config_name, results in all_ablation_results.items():
    em_all = [em_score(r['prediction'],r['gold']) for r in results]
    f1_all = [f1_calc(r['prediction'],r['gold']) for r in results]
    br = [r for r in results if r['type']=='bridge']
    co = [r for r in results if r['type']=='comparison']
    br_em = sum(em_score(r['prediction'],r['gold']) for r in br)/max(len(br),1)*100
    br_f1 = sum(f1_calc(r['prediction'],r['gold']) for r in br)/max(len(br),1)*100
    co_em = sum(em_score(r['prediction'],r['gold']) for r in co)/max(len(co),1)*100
    co_f1 = sum(f1_calc(r['prediction'],r['gold']) for r in co)/max(len(co),1)*100
    overall_em = sum(em_all)/len(em_all)*100
    overall_f1 = sum(f1_all)/len(f1_all)*100
    desc = ABLATION_CONFIGS.get(config_name, {}).get('desc', config_name)
    marker = ' ★' if overall_em > 26.0 else ''
    print(f'{config_name:<35} {overall_em:>7.1f}% {overall_f1:>7.1f}% {br_em:>7.1f}% {br_f1:>7.1f}% {co_em:>7.1f}% {co_f1:>7.1f}%{marker}')
    nl = [r['nodes_traversed'] for r in results]
    la = [r['latency_seconds'] for r in results]
    summary_metrics[config_name] = {
        'desc': desc,
        'overall_em': round(overall_em/100,4), 'overall_f1': round(overall_f1/100,4),
        'bridge_em': round(br_em/100,4), 'bridge_f1': round(br_f1/100,4),
        'comparison_em': round(co_em/100,4), 'comparison_f1': round(co_f1/100,4),
        'avg_nodes': round(sum(nl)/len(nl),1), 'avg_latency_s': round(sum(la)/len(la),3),
    }

print(f'{"="*70}')
print('★ = beats Flat RAG baseline (26.0% EM)')
print('\nK-sensitivity summary (all at depth=1, 0.5/0.5):')
print('  k=2:', summary_metrics.get('K2_depth1', {}).get('overall_em', '?'))
print('  k=3:', summary_metrics.get('K3_depth1', {}).get('overall_em', '?'))
print('  k=5: 0.2500 (from previous Config B)')

# Save
with open(DRIVE_OUTPUTS+'ablation_k_sensitivity_summary.json','w') as f:
    json.dump({'baselines': baselines, 'k_sensitivity': summary_metrics}, f, indent=2)
print(f'\nSaved to {DRIVE_OUTPUTS}ablation_k_sensitivity_summary.json')
print('\n✅ K-sensitivity analysis complete!')